# 30 · Phase H1f — Paired Same-Record SHARP Keyword Parity Audit

This notebook queries `hmi.sharp_720s` and `hmi.sharp_cea_720s` for the **same HARP and same T_REC** and compares the 16 frozen SHARP summary keywords directly.

It downloads no image segments and changes no forecast model or routing policy.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, time
import numpy as np
import pandas as pd

try:
    import drms
except Exception as exc:
    raise RuntimeError(
        "The 'drms' package is required for this metadata-only JSOC audit. "
        "Install it into /home/abmoses2000/aia_gpu_venv and rerun."
    ) from exc

ROOT=Path("/home/abmoses2000")
SOURCE_ROOT=ROOT/"aia_sharp16_refit9_evaluation_preparation_20261002"/"inputs"/"sharp16_matched_v1_20261001"
NATIVE=SOURCE_ROOT/"supplementary_2026"/"native_slot_ledger.csv.gz"
INDEX=SOURCE_ROOT/"supplementary_2026"/"index.csv.gz"
OUT=ROOT/"graybox_aia72_sharp_keyword_parity_h1f_v1_20261006"

for p in [NATIVE,INDEX]:
    assert p.is_file(), f"Missing required source: {p}"
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

FEATURES=["MEANGBZ","MEANGAM","MEANGBT","MEANGBH","MEANJZD","TOTUSJZ","MEANALP","MEANJZH",
          "ABSNJZH","SAVNCPP","MEANSHR","SHRGT45","R_VALUE","USFLUX","TOTPOT","TOTUSJH"]
META=["HARPNUM","T_REC","DATE","CODEVER7","CALVER64","CMASK","QUALITY"]
KEYS=META+FEATURES

native=pd.read_csv(NATIVE,low_memory=False)
idx=pd.read_csv(INDEX,low_memory=False)

# Primary records are latest slots only because H1d identified latest-state gradients as dominant.
latest=native[native["slot"].eq(2)].copy()
latest["T_REC_dt"]=pd.to_datetime(latest["T_REC_TAI"].astype(str).str.replace("_TAI","",regex=False),
                                  format="%Y.%m.%d_%H:%M:%S",utc=True,errors="raise")
latest["month"]=latest["T_REC_dt"].dt.to_period("M").astype(str)

print("drms",drms.__version__)
print("latest-slot records:",len(latest))


In [ ]:
# Deterministic, outcome-free sample:
# 8 evenly spaced latest-slot rows per month + records nearest the source transition ±12h.
parts=[]
for month,g in latest.groupby("month",sort=True):
    g=g.sort_values(["T_REC_dt","HARPNUM","target_sample_id"],kind="stable").reset_index(drop=True)
    n=min(8,len(g))
    take=np.unique(np.linspace(0,len(g)-1,n,dtype=int))
    parts.append(g.iloc[take])

transition=pd.Timestamp("2026-04-04 00:48:00",tz="UTC")
near=latest[(latest["T_REC_dt"]>=transition-pd.Timedelta(hours=12)) &
            (latest["T_REC_dt"]<=transition+pd.Timedelta(hours=12))].copy()
near=near.sort_values(["T_REC_dt","HARPNUM"],kind="stable")
parts.append(near)

sample=(pd.concat(parts,ignore_index=True)
        .drop_duplicates(["HARPNUM","T_REC_TAI"])
        .sort_values(["T_REC_dt","HARPNUM"],kind="stable")
        .reset_index(drop=True))

sample[["target_sample_id","HARPNUM","T_REC_TAI","month","series","source_origin"]].to_csv(
    OUT/"paired_query_sample.csv",index=False)

print("Deterministic sample records:",len(sample))
print(sample.groupby("month").size().to_string())
print("\nTransition-window records:",len(near))


In [ ]:
client=drms.Client()
series_list=["hmi.sharp_720s","hmi.sharp_cea_720s"]
rows=[]

for i,r in sample.iterrows():
    harp=int(r["HARPNUM"])
    trec=str(r["T_REC_TAI"])
    for series in series_list:
        q=f"{series}[{harp}][{trec}]"
        rec={
          "sample_index":int(i),
          "target_sample_id":str(r["target_sample_id"]),
          "HARPNUM_requested":harp,
          "T_REC_requested":trec,
          "series_requested":series,
          "query":q,
          "query_ok":False,
          "rows_returned":0,
          "error":"",
        }
        try:
            ans=client.query(q,key=",".join(KEYS))
            rec["rows_returned"]=int(len(ans))
            if len(ans)==1:
                rec["query_ok"]=True
                for c in KEYS:
                    rec[c]=ans.iloc[0][c] if c in ans.columns else np.nan
            elif len(ans)==0:
                rec["error"]="no_record"
            else:
                rec["error"]=f"multiple_records:{len(ans)}"
        except Exception as exc:
            rec["error"]=repr(exc)
        rows.append(rec)
        time.sleep(0.05)

queried=pd.DataFrame(rows)
queried.to_csv(OUT/"paired_jsoc_keyword_queries.csv.gz",index=False,
               compression={"method":"gzip","mtime":0})

print("Queries:",len(queried))
print("Successful:",int(queried["query_ok"].sum()))
print("Failures:")
print(queried.loc[~queried.query_ok,["series_requested","HARPNUM_requested","T_REC_requested","error"]].head(30).to_string(index=False))


In [ ]:
# Reshape successful same-record pairs and compare numeric keywords.
ok=queried[queried["query_ok"]].copy()
pair_counts=ok.groupby("sample_index")["series_requested"].nunique()
paired_ids=pair_counts[pair_counts.eq(2)].index
paired=ok[ok["sample_index"].isin(paired_ids)].copy()

a=paired[paired.series_requested.eq("hmi.sharp_720s")].set_index("sample_index")
b=paired[paired.series_requested.eq("hmi.sharp_cea_720s")].set_index("sample_index")
common=a.index.intersection(b.index)
assert len(common)==len(paired_ids)

diff_rows=[]
for sid in common:
    for f in FEATURES:
        va=pd.to_numeric(pd.Series([a.loc[sid,f]]),errors="coerce").iloc[0]
        vb=pd.to_numeric(pd.Series([b.loc[sid,f]]),errors="coerce").iloc[0]
        finite=np.isfinite(va) and np.isfinite(vb)
        absdiff=float(abs(vb-va)) if finite else np.nan
        scale=max(abs(float(va)),abs(float(vb)),1e-12) if finite else np.nan
        reldiff=float(absdiff/scale) if finite else np.nan
        diff_rows.append({
          "sample_index":int(sid),
          "target_sample_id":a.loc[sid,"target_sample_id"],
          "HARPNUM":int(a.loc[sid,"HARPNUM_requested"]),
          "T_REC":a.loc[sid,"T_REC_requested"],
          "feature":f,
          "ccd_value":va,
          "cea_value":vb,
          "absolute_difference":absdiff,
          "relative_difference":reldiff,
          "numeric_equal_1e12":bool(np.isclose(va,vb,rtol=1e-12,atol=1e-12,equal_nan=True)) if finite else False,
          "numeric_equal_1e9":bool(np.isclose(va,vb,rtol=1e-9,atol=1e-12,equal_nan=True)) if finite else False,
        })

diff=pd.DataFrame(diff_rows)
diff.to_csv(OUT/"paired_feature_differences.csv.gz",index=False,
            compression={"method":"gzip","mtime":0})

feature_summary=(diff.groupby("feature").agg(
    paired_records=("sample_index","nunique"),
    exact_numeric_1e12_rate=("numeric_equal_1e12","mean"),
    numeric_1e9_rate=("numeric_equal_1e9","mean"),
    max_abs_difference=("absolute_difference","max"),
    median_abs_difference=("absolute_difference","median"),
    max_relative_difference=("relative_difference","max"),
).reset_index())
feature_summary.to_csv(OUT/"paired_feature_parity_summary.csv",index=False)

# Metadata parity (excluding DATE because archive processing timestamp may differ).
meta_rows=[]
for c in ["HARPNUM","T_REC","CODEVER7","CALVER64","CMASK","QUALITY"]:
    equal=[]
    for sid in common:
        va=str(a.loc[sid,c]); vb=str(b.loc[sid,c])
        equal.append(va==vb)
    meta_rows.append({"field":c,"paired_records":len(common),"exact_string_equal_rate":float(np.mean(equal))})
meta=pd.DataFrame(meta_rows)
meta.to_csv(OUT/"paired_metadata_parity_summary.csv",index=False)

print("===== FEATURE PARITY =====")
print(feature_summary.to_string(index=False))
print("\n===== METADATA PARITY =====")
print(meta.to_string(index=False))


In [ ]:
# Automatic descriptive classification; no policy action.
all_success=(queried["query_ok"].mean()==1.0)
pair_fraction=len(common)/len(sample) if len(sample) else 0.0
all_features_1e9=bool((feature_summary["numeric_1e9_rate"]==1.0).all()) if len(feature_summary) else False

if pair_fraction>=0.95 and all_features_1e9:
    interpretation="SUMMARY_KEYWORDS_PARITY_CONFIRMED"
elif pair_fraction>=0.80 and (feature_summary["numeric_1e9_rate"]<0.95).any():
    interpretation="SUMMARY_KEYWORDS_DIFFER_BY_SERIES"
elif pair_fraction>0:
    interpretation="PARTIAL_PARITY_OR_RECORD_GAPS"
else:
    interpretation="JSOC_QUERY_UNAVAILABLE"

receipt={
 "status":"PHASE_H1F_PAIRED_SHARP_KEYWORD_AUDIT_COMPLETE",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "policy_changed":False,
 "metadata_only_jsoc_query":True,
 "sample_records":int(len(sample)),
 "successful_series_queries":int(queried["query_ok"].sum()),
 "paired_same_record_comparisons":int(len(common)),
 "paired_fraction":float(pair_fraction),
 "interpretation":interpretation,
 "source_sha256":{
   "supplementary_native_slot_ledger":hashlib.sha256(Path(NATIVE).read_bytes()).hexdigest(),
   "supplementary_index":hashlib.sha256(Path(INDEX).read_bytes()).hexdigest(),
 },
 "policy_action":"NONE",
}
(OUT/"h1f_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps(receipt,indent=2))
